### DESC, QA generation

In [ ]:
import os
import ast
import json
import random
from scene_annot import extract_annotation, get_single_scene_qas, get_mixed_scene_qa
from utils import concat

DATA_DIR = '../scene'
OUTPUT_DIR = '../../data/qwen2.5-vl'
MIXED_DIR = 'mixed3'

TEST_RATIO = 0.1

SAMPLE_NUM_PER_QTYPE = 30
VISCOSITY_TYPE_NUM = 100
# total test QAs = 30 * (7 + 1) + 100 = 340

ANNOT_DIRS = {
    'free_fall': ['free_fall/output', 'free_fall/output2', 'free_fall/output3'],
    'slope': ['slope/output'],
    'ripple': ['ripple/output', 'ripple/output3'],
    'obj_moving': ['obj_moving/output', 'obj_moving/output3'],
    'obj_interaction': ['obj_interaction/output'],
}

random.seed(42)

os.makedirs(OUTPUT_DIR, exist_ok=True)

total_test_entries = []
for scene_type in ANNOT_DIRS.keys():
    entries = []
    for subfolder in ANNOT_DIRS[scene_type]:
        i = 0
        sub_path = os.path.join(DATA_DIR, subfolder)
        while True:
            log_path = os.path.join(sub_path, str(i), "params.log")
            if not os.path.isfile(log_path):
                break

            with open(log_path, "r", encoding="utf-8") as f:
                content = f.read().strip()
                try:
                    params_dict = ast.literal_eval(content)
                except (ValueError, SyntaxError):
                    try:
                        params_dict = json.loads(content)
                    except json.JSONDecodeError:
                        print(f"Warning: cannot parse '{log_path}'.")
                        break
                    
            entries.append({
                'scene_type': scene_type,
                'video': f"{subfolder}/{i}/render.mkv", 
                'params': params_dict
            })
            i += 1
        
    random.shuffle(entries)
    test_size = int(len(entries) * TEST_RATIO)
    train_entries = entries[:-test_size]
    test_entries = entries[-test_size:]
    total_test_entries.extend(test_entries)
    
    ### Save train descriptions
    descs = []
    for entry in train_entries:
        try:
            annotation = extract_annotation(entry['params'], scene_type)
        except Exception as e:
            print(f"Error extracting annotation for {entry['video']}: {e}")
            print(f"Skipping entry: {entry}")
            continue
        desc = {
            'scene_type': scene_type,
            'video': entry['video'],
            'conversations': [
                {
                    "from": "human",
                    "value": f"<video>\nDescribe the video."
                },
                {
                    "from": "gpt",
                    "value": annotation
                }
            ]
        }
        descs.append(desc)
    
    os.makedirs(os.path.join(OUTPUT_DIR, scene_type), exist_ok=True)    
    train_file = os.path.join(OUTPUT_DIR, scene_type, f"train.json")
    with open(train_file, "w", encoding="utf-8") as f:
        json.dump(descs, f, ensure_ascii=False, indent=4)
        
    print(f"Saved {len(descs)} training entries to {train_file}")
    
### Generate test QAs
total_test_qas = []

Q_TYPE_LIST = ['obj_color', 'obj_rot', 'obj_loc', 'fluid_color', 'fluid_direction', 'fluid_loc', 'collision', 'fluid_amount', 'fluid_viscosity']

# single scene test QAs
def format_options(options):
    return ", ".join([f"{i+1}. {opt}" for i, opt in enumerate(options)])

for entry in total_test_entries:
    scene_type = entry['scene_type']
    video = entry['video']
    params = entry['params']
    
    qas = get_single_scene_qas(params, scene_type)

    qas = [{
        'q_type': qa['q_type'],
        'video': video,
        'conversations': [
            {
                "from": "human",
                "value": '<video>\n' + qa['question'] + '\nOptions: ' + format_options(qa['options'])
            },
            {
                "from": "gpt",
                "value": f"{qa['answer'] + 1}. {qa['options'][qa['answer']]}"
            }
        ]
    } for qa in qas]
    
    total_test_qas.extend(qas)

# mixed scene test QAs
os.makedirs(os.path.join(DATA_DIR, MIXED_DIR), exist_ok=True)
# fluid amount
free_fall_entries = [entry for entry in total_test_entries if entry['scene_type'] == 'free_fall']
ripple_entries = [entry for entry in total_test_entries if entry['scene_type'] == 'ripple']
for i in range(SAMPLE_NUM_PER_QTYPE // 2):
    free_fall_scenes = random.sample(free_fall_entries, 2)
    ripple_scenes = random.sample(ripple_entries, 2)

    free_fall_qa = get_mixed_scene_qa(free_fall_scenes[0]['params'], free_fall_scenes[1]['params'], 'free_fall')
    ripple_qa = get_mixed_scene_qa(ripple_scenes[0]['params'], ripple_scenes[1]['params'], 'ripple')

    concat(os.path.join(DATA_DIR, free_fall_scenes[0]['video']), os.path.join(DATA_DIR, free_fall_scenes[1]['video']), os.path.join(DATA_DIR, MIXED_DIR, f'free_fall_{i}.mkv')) 
    concat(os.path.join(DATA_DIR, ripple_scenes[0]['video']), os.path.join(DATA_DIR, ripple_scenes[1]['video']), os.path.join(DATA_DIR, MIXED_DIR, f'ripple_{i}.mkv'))

    total_test_qas.append({
        'q_type': 'fluid_amount',
        'video_list': [free_fall_scenes[0]['video'], free_fall_scenes[1]['video']],
        'video': f'{MIXED_DIR}/free_fall_{i}.mkv',
        'conversations': [
            {
                "from": "human",
                "value": '<video>\n' + free_fall_qa['question'] + '\nOptions: ' + format_options(free_fall_qa['options'])
            },
            {
                "from": "gpt",
                "value": f"{free_fall_qa['answer'] + 1}. {free_fall_qa['options'][free_fall_qa['answer']]}"
            }
        ]
    })
    total_test_qas.append({
        'q_type': 'fluid_amount',
        'video_list': [ripple_scenes[0]['video'], ripple_scenes[1]['video']],
        'video': f'{MIXED_DIR}/ripple_{i}.mkv',
        'conversations': [
            {
                "from": "human",
                "value": '<video>\n' + ripple_qa['question'] + '\nOptions: ' + format_options(ripple_qa['options'])
            },
            {
                "from": "gpt",
                "value": f"{ripple_qa['answer'] + 1}. {ripple_qa['options'][ripple_qa['answer']]}"
            }
        ]
    })
# fluid viscosity
cnt = 0
while cnt < VISCOSITY_TYPE_NUM:
    scenes = random.sample(total_test_entries, 2)
    if scenes[0]['scene_type'] == 'obj_interaction' or scenes[1]['scene_type'] == 'obj_interaction':
        if scenes[0]['scene_type'] != scenes[1]['scene_type']:
            continue
    
    qa = get_mixed_scene_qa(scenes[0]['params'], scenes[1]['params'], 'mixed')
    if qa is None:
        continue

    concat(os.path.join(DATA_DIR, scenes[0]['video']), os.path.join(DATA_DIR, scenes[1]['video']), os.path.join(DATA_DIR, MIXED_DIR, f'viscosity_{cnt}.mkv'))

    total_test_qas.append({
        'q_type': 'fluid_viscosity',
        'video_list': [scenes[0]['video'], scenes[1]['video']],
        'video': f'{MIXED_DIR}/viscosity_{cnt}.mkv',
        'conversations': [
            {
                "from": "human",
                "value": '<video>\n' + qa['question'] + '\nOptions: ' + format_options(qa['options'])
            },
            {
                "from": "gpt",
                "value": f"{qa['answer'] + 1}. {qa['options'][qa['answer']]}"
            }
        ]
    })

    cnt += 1

# 개수 조정 (각 Q_TYPE_LIST에 대해 SAMPLE_NUM_PER_QTYPE개씩 랜덤 추출)
random.shuffle(total_test_qas)
final_test_qas = []
for q_type in Q_TYPE_LIST:
    qas_of_type = [qa for qa in total_test_qas if qa['q_type'] == q_type]
    if q_type == 'fluid_viscosity':
        # fluid_viscosity는 VISCOSITY_TYPE_NUM으로 제한
        selected_qas = random.sample(qas_of_type, min(VISCOSITY_TYPE_NUM, len(qas_of_type)))
    else:
        # 나머지 Q_TYPE은 SAMPLE_NUM_PER_QTYPE으로 제한
        selected_qas = random.sample(qas_of_type, min(SAMPLE_NUM_PER_QTYPE, len(qas_of_type)))
    final_test_qas.extend(selected_qas)

# Save test QAs
test_file = os.path.join(OUTPUT_DIR, 'test1.json')
with open(test_file, "w", encoding="utf-8") as f:
    json.dump(final_test_qas, f, ensure_ascii=False, indent=4)
print(f"Saved {len(final_test_qas)} test QAs to {test_file}")




Saved 1296 training entries to ../../data/qwen2.5-vl/free_fall/train.json
Saved 317 training entries to ../../data/qwen2.5-vl/slope/train.json
Saved 227 training entries to ../../data/qwen2.5-vl/ripple/train.json
Error extracting annotation for ripple/output3/216/render.mkv: 'color'
Skipping entry: {'scene_type': 'obj_moving', 'video': 'ripple/output3/216/render.mkv', 'params': {'cam_loc': [3.9416, -2.8738, 3.1844], 'cam_rot': [0.9358, 0.0, 0.9176], 'viscosity': -1.0, 'part_rad': 0.8, 'part_num': 2, 'part_random': 0.1, 'ts_min': 1, 'flow_v': [0, 0, 0], 'water_loc': [-0.1745, 1.535, 2.2169], 'water_size': 2.0, 'light_angle': [0.3963, 0.127, -0.7037], 'water_color': [0.3427, 0.2505, 0.1456, 1.0], 'water_alpha': 0.995, 'samples': 16, 'step': 3, '_cache_dir_name': 'b4e71b9a731698ac4c6f7dae28f3214a762a3288_n_0_1_0_n_n_n_n'}}
Error extracting annotation for ripple/output3/115/render.mkv: 'color'
Skipping entry: {'scene_type': 'obj_moving', 'video': 'ripple/output3/115/render.mkv', 'params': 

frame_index:  58%|█████▊    | 29/50 [00:04<00:03,  6.36it/s, now=None]

KeyboardInterrupt: 

### Multi Scene (Deprecated)

In [ ]:
import os
import ast
import json
import random
from scene_annot import extract_annotation
from utils import concat

DATA_DIR = '../scene'
OUTPUT_DIR = '../../data/qwen2.5-vl'

TEST_RATIO = 0.1

ANNOT_DIRS = {
    'free_fall': ['free_fall/output'],
    'slope': [],
    'ripple': [],
    'obj_moving': ['obj_moving/output'],
    'obj_interaction': []
}

params = []

for scene_type in ANNOT_DIRS.keys():
    for subfolder in ANNOT_DIRS[scene_type]:
        sub_path = os.path.join(DATA_DIR, subfolder)
        i = 0
        while True:
            log_path = os.path.join(sub_path, str(i), "params.log")
            if not os.path.isfile(log_path):
                break

            with open(log_path, "r", encoding="utf-8") as f:
                content = f.read().strip()
                try:
                    params_dict = ast.literal_eval(content)
                except (ValueError, SyntaxError):
                    try:
                        params_dict = json.loads(content)
                    except json.JSONDecodeError:
                        print(f"Warning: cannot parse '{log_path}'.")
                        break

            params.append({
                "scene_type": scene_type,
                "video": f"{subfolder}/{i}/render.mkv",
                "params": params_dict
            })
            i += 1

print(f"Total scenes: {len(params)}")

TRAIN_SIZE = int(len(params) * (1 - TEST_RATIO))

params = random.sample(params, len(params))  # Shuffle the parameters
train_params = params[:TRAIN_SIZE]
test_params = params[TRAIN_SIZE:]

TRAIN_SAMPLE_SIZE = 1000
TEST_SAMPLE_SIZE = 100

train_descs, test_descs = [], []
train_qas, test_qas = [], []
descs, qas = [], []

os.makedirs(f"{DATA_DIR}/concat", exist_ok=True)
os.makedirs(f"{DATA_DIR}/concat/train", exist_ok=True)
os.makedirs(f"{DATA_DIR}/concat/test", exist_ok=True)

for i in range(TRAIN_SAMPLE_SIZE):
    choices = random.sample(train_params, 2)
    os.makedirs(f"{DATA_DIR}/concat/train/{i}", exist_ok=True)

    concat(f"{DATA_DIR}/{choices[0]['video']}", f"{DATA_DIR}/{choices[1]['video']}", f"{DATA_DIR}/concat/train/{i}/row.mkv", f"{DATA_DIR}/concat/train/{i}/col.mkv")

    annot1 = extract_annotation(choices[0]['params'], choices[0]['scene_type'])
    annot2 = extract_annotation(choices[1]['params'], choices[1]['scene_type'])

    for e in ['row', 'col']:
        train_descs.append({
            "video": f"concat/train/{i}/{e}.mkv",
            "conversations": [
                {
                    "from": "human",
                    "value": f"<video>\nDescribe the video."
                },
                {
                    "from": "gpt",
                    "value": f"First Scene: {annot1}\nSecond Scene: {annot2}"
                }
            ]
        })
    
    visco1 = choices[0]['params']['viscosity']
    visco2 = choices[1]['params']['viscosity']
    visco1 = 0 if visco1 < 0.001 else 1 if visco1 < 0.01 else 2
    visco2 = 0 if visco2 < 0.001 else 1 if visco2 < 0.01 else 2

    if visco1 == visco2:
        continue
    else:
        for e in ['row', 'col']:
            train_qas.append({
                "video": f"concat/train/{i}/{e}.mkv",
                "conversations": [
                    {
                        "from": "human",
                        "value": f"<video>\nWhich scene has higher viscosity? First or Second?"
                    },
                    {
                        "from": "gpt",
                        "value": "The first scene." if visco1 > visco2 else "The second scene."
                    }
                ]
            })

for i in range(TEST_SAMPLE_SIZE):
    choices = random.sample(test_params, 2)
    os.makedirs(f"{DATA_DIR}/concat/test/{i}", exist_ok=True)
    
    concat(f"{DATA_DIR}/{choices[0]['video']}", f"{DATA_DIR}/{choices[1]['video']}", f"{DATA_DIR}/concat/test/{i}/row.mkv", f"{DATA_DIR}/concat/test/{i}/col.mkv")

    annot1 = extract_annotation(choices[0]['params'], choices[0]['scene_type'])
    annot2 = extract_annotation(choices[1]['params'], choices[1]['scene_type'])

    for e in ['row', 'col']:
        test_descs.append({
            "video": f"concat/test/{i}/{e}.mkv",
            "conversations": [
                {
                    "from": "human",
                    "value": f"<video>\nDescribe the video."
                },
                {
                    "from": "gpt",
                    "value": f"First Scene: {annot1}\nSecond Scene: {annot2}"
                }
            ]
        })
    
    visco1 = choices[0]['params']['viscosity']
    visco2 = choices[1]['params']['viscosity']
    visco1 = 0 if visco1 < 0.001 else 1 if visco1 < 0.01 else 2
    visco2 = 0 if visco2 < 0.001 else 1 if visco2 < 0.01 else 2

    if visco1 == visco2:
        continue
    else:
        for e in ['row', 'col']:
            test_qas.append({
                "video": f"concat/test/{i}/{e}.mkv",
                "conversations": [
                    {
                        "from": "human",
                        "value": f"<video>\nWhich scene has higher viscosity? First or Second?"
                    },
                    {
                        "from": "gpt",
                        "value": "The fi"
                        "rst scene." if visco1 > visco2 else "The second scene."
                    }
                ]
            })


# Save the results
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
if not os.path.exists(os.path.join(OUTPUT_DIR, "concat")):
    os.makedirs(os.path.join(OUTPUT_DIR, "concat"))

with open(os.path.join(OUTPUT_DIR, "concat/desc_train.json"), "w", encoding="utf-8") as f:
    json.dump(train_descs, f, indent=4, ensure_ascii=False)
with open(os.path.join(OUTPUT_DIR, "concat/desc_test.json"), "w", encoding="utf-8") as f:
    json.dump(test_descs, f, indent=4, ensure_ascii=False)
with open(os.path.join(OUTPUT_DIR, "concat/qa_train.json"), "w", encoding="utf-8") as f:
    json.dump(train_qas, f, indent=4, ensure_ascii=False)
with open(os.path.join(OUTPUT_DIR, "concat/qa_test.json"), "w", encoding="utf-8") as f:
    json.dump(test_qas, f, indent=4, ensure_ascii=False)

print(f"Processed concatenated scenes: {len(train_descs)} train descriptions, {len(test_descs)} test descriptions.")
print(f"Processed concatenated scenes: {len(train_qas)} train QAs, {len(test_qas)} test QAs.")


In [ ]:
# json file lists in OUTPUT_DIR/concat
files = os.listdir(os.path.join(OUTPUT_DIR, "concat"))

for file in files:
    new_data = []
    with open(os.path.join(OUTPUT_DIR, "concat", file), "r", encoding="utf-8") as f:
        data = json.load(f)
        print(f"{file}: {len(data)} entries")
        for entry in data:
            if 'test' in file:
                entry['video'] = 'concat/test/' + entry['video'].replace('concat/', '')
            elif 'train' in file:
                entry['video'] = 'concat/train/' + entry['video'].replace('concat/', '')
            new_data.append(entry)
    with open(os.path.join(OUTPUT_DIR, "concat", file), "w", encoding="utf-8") as f:
        json.dump(new_data, f, indent=4, ensure_ascii=False)
print("Updated video paths in concatenated JSON files.")


    

